# Working with raw MultispeQ traces

Most datasets include a `sample_raw` column containing the **full raw measurement trace** from the MultispeQ device, stored as a Databricks VARIANT type. This column is excluded by the default `src.data` loaders because VARIANT is not supported by Databricks Connect / pandas.

This notebook shows how to access and parse the raw traces directly via PySpark SQL.

**Parsing approach:** VARIANT types cannot be indexed directly (e.g. `sample_raw[0]:set` returns NULL). Instead, cast the entire column to STRING and parse it with `from_json()`:

```sql
from_json(CAST(sample_raw AS STRING), 'ARRAY<STRUCT<...>>') AS parsed
-- then access: parsed[0].set[3].temperature
```

**Structure of `sample_raw`:**

Each value is a JSON array containing one protocol run object:

```json
[{
  "protocol_id": "3517",
  "set": [
    {"label": "start_time", "data_raw": [], ...},
    {"label": "env", "temperature": 26.86, "humidity": 69.1, ...},
    {"label": "PIRK", "data_raw": [48907.0, 48919.0, ...], ...},
    {"label": "PAM-ABS", "data_raw": [5678, 5679, ...], ...},
    ...
  ],
  "set_repeats": 2,
  "v_arrays": [...]
}]
```

The `set` array contains labeled measurement steps. Key labels:

| Label        | Contents                                                                             |
| ------------ | ------------------------------------------------------------------------------------ |
| `env`        | Environmental readings (temperature, humidity, pressure, light, compass, leaf angle) |
| `autogain`   | Detector calibration values                                                          |
| `PIRK`       | Post-Illumination Rise in fluorescence Kinetics trace (`data_raw`)                   |
| `PAM-ABS`    | Pulse-Amplitude Modulation fluorescence trace (`data_raw`)                           |
| `PAM-ABS-FR` | Far-red fluorescence trace (`data_raw`)                                              |
| `SPAD`       | Chlorophyll content measurement                                                      |

**Datasets with `sample_raw`:** all except Cowpea IITA (08).

**Requires:** Databricks (PySpark with VARIANT support).


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.connect.session import SparkSession

CATALOG = "open_jii_data_hackathon.default"

# Schema for from_json(): only the fields we need from each set element.
# The full VARIANT schema has many more fields, but from_json silently
# ignores anything not declared here.
SET_SCHEMA = "ARRAY<STRUCT<label:STRING, temperature:DOUBLE, humidity:DOUBLE, pressure:DOUBLE, light_intensity:DOUBLE, contactless_temp:DOUBLE, angle:DOUBLE, compass_direction:STRING, data_raw:ARRAY<DOUBLE>>>"
RAW_SCHEMA = f"ARRAY<STRUCT<protocol_id:STRING, set:{SET_SCHEMA}, set_repeats:BIGINT>>"

spark = SparkSession.getActiveSession()
assert spark is not None, "This notebook must be run on Databricks"

## Extracting environmental data from raw traces

The `env` set (typically `set[3]`) contains per-measurement environmental readings that may not all be in the computed columns.


In [ ]:
# Extract environmental readings from sample_raw
# Step 1: cast VARIANT to STRING, then parse with from_json
# Step 2: access parsed[0].set[3] (where label='env')
env = spark.sql(
    f"""
    SELECT
        measurement_id,
        parsed[0].set[3].label AS label,
        parsed[0].set[3].temperature AS temperature,
        parsed[0].set[3].humidity AS humidity,
        parsed[0].set[3].pressure AS pressure,
        parsed[0].set[3].light_intensity AS light_intensity,
        parsed[0].set[3].contactless_temp AS leaf_temp,
        parsed[0].set[3].angle AS leaf_angle,
        parsed[0].set[3].compass_direction AS compass_dir
    FROM (
        SELECT measurement_id,
               from_json(CAST(sample_raw AS STRING), '{RAW_SCHEMA}') AS parsed
        FROM {CATALOG}.grebbedijk_measurements
        WHERE sample_raw IS NOT NULL
    )
    LIMIT 10
"""
)

env.show()

## Extracting fluorescence traces

The PIRK and PAM-ABS sets contain `data_raw` arrays — the actual time-resolved fluorescence signals. These can be exploded into individual data points.


In [ ]:
# Extract PIRK trace from the first measurement
# PIRK is typically set[5] (ambient) and set[17] (high light)
pirk = spark.sql(
    f"""
    SELECT
        measurement_id,
        parsed[0].set[5].label AS label,
        t.time_index,
        t.signal
    FROM (
        SELECT measurement_id,
               from_json(CAST(sample_raw AS STRING), '{RAW_SCHEMA}') AS parsed
        FROM {CATALOG}.grebbedijk_measurements
        WHERE sample_raw IS NOT NULL
        LIMIT 1
    ),
    LATERAL (
        SELECT POSEXPLODE(parsed[0].set[5].data_raw) AS (time_index, signal)
    ) t
"""
)

print(f"PIRK trace points: {pirk.count()}")
pirk.show(20)

In [ ]:
# Extract PAM-ABS fluorescence trace
# PAM-ABS is typically set[6] (ambient) and set[18] (high light)
pam = spark.sql(
    f"""
    SELECT
        measurement_id,
        parsed[0].set[6].label AS label,
        t.time_index,
        t.signal
    FROM (
        SELECT measurement_id,
               from_json(CAST(sample_raw AS STRING), '{RAW_SCHEMA}') AS parsed
        FROM {CATALOG}.grebbedijk_measurements
        WHERE sample_raw IS NOT NULL
        LIMIT 1
    ),
    LATERAL (
        SELECT POSEXPLODE(parsed[0].set[6].data_raw) AS (time_index, signal)
    ) t
"""
)

print(f"PAM-ABS trace points: {pam.count()}")
pam.show(20)

## Finding set labels dynamically

The set index for each label can vary between protocols. Here's how to find which index corresponds to which label.


In [ ]:
# List all set labels and their indices for one measurement
labels = spark.sql(
    f"""
    SELECT
        pos AS set_index,
        col.label AS label,
        SIZE(col.data_raw) AS data_raw_len
    FROM (
        SELECT POSEXPLODE(parsed[0].set)
        FROM (
            SELECT from_json(CAST(sample_raw AS STRING), '{RAW_SCHEMA}') AS parsed
            FROM {CATALOG}.grebbedijk_measurements
            WHERE sample_raw IS NOT NULL
            LIMIT 1
        )
    )
"""
)

labels.show(30, truncate=False)

## Batch: extract traces for multiple measurements

To compare fluorescence kinetics across genotypes or conditions, extract traces for many measurements at once.


In [ ]:
# Extract PIRK traces for multiple measurements, with genotype info
batch = spark.sql(
    f"""
    SELECT
        m.measurement_id,
        m.Genotype,
        t.time_index,
        t.signal
    FROM (
        SELECT measurement_id, Genotype,
               from_json(CAST(sample_raw AS STRING), '{RAW_SCHEMA}') AS parsed
        FROM {CATALOG}.grebbedijk_measurements
        WHERE sample_raw IS NOT NULL
    ) m,
    LATERAL (
        SELECT POSEXPLODE(m.parsed[0].set[5].data_raw) AS (time_index, signal)
    ) t
    LIMIT 5000
"""
)

print(f"Total trace points: {batch.count():,}")
batch.show(20)

## Other datasets

The same approach works for any dataset with `sample_raw`. Just change the table name.
The set indices will differ by protocol — use the label discovery query above to find them.

| Table                       | Protocol       | Key trace labels          |
| --------------------------- | -------------- | ------------------------- |
| `grebbedijk_measurements`   | UNZA_PIRK_DIRK | PIRK, PAM-ABS, PAM-ABS-FR |
| `bean_gart_35462_35509`     | UNZA_PIRK_DIRK | PIRK, PAM-ABS, PAM-ABS-FR |
| `barley_qtl_32593_32742`    | UNZA_PIRK_DIRK | PIRK, PAM-ABS, PAM-ABS-FR |
| `aardaker_nergena_29273`    | RIDES 2.0      | ECS, P700, PAM, ABS       |
| `barley_imagic_17237_18685` | RIDES 2.0      | ECS, P700, PAM, ABS       |
| `barley_hvdrr_12922_16934`  | RIDES + PSII   | Mixed (check per project) |
